# Physics-Informed Solar Energy Analytics

**Dataset notice:** all records are synthetic and generated for learning. They are not real PV plant measurements.

## 1. Load and inspect data

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
if not (ROOT/'data').exists(): ROOT = ROOT.parent
df = pd.read_csv(ROOT/'data'/'synthetic_solar_pv_hourly.csv', parse_dates=['timestamp'])
df.head()

In [ ]:
df.info()
df.isna().sum()

In [ ]:
df[['irradiance_w_m2','module_temperature_c','expected_power_kw','actual_power_kw','performance_ratio']].describe()

## 2. Explore output and injected fault labels

In [ ]:
import matplotlib.pyplot as plt
sample=df.iloc[:168]
plt.figure(figsize=(12,4))
plt.plot(sample.timestamp,sample.actual_power_kw,label='Actual simulated output')
plt.plot(sample.timestamp,sample.expected_power_kw,label='Physics-inspired expected output')
plt.ylabel('Power (kW)'); plt.xticks(rotation=30); plt.legend(); plt.tight_layout()

In [ ]:
df.fault_type.value_counts()

## 3. Run the reproducible modelling workflow
The script writes forecasting and fault-classification metrics, confusion matrix and plots under `reports/`.

In [ ]:
import sys
sys.path.insert(0,str(ROOT/'src'))
from run_analysis import run
run()

## 4. Inspect metrics and critically evaluate them
Do not interpret synthetic performance as real-world accuracy. In particular, the fault classifier uses derived performance variables that may encode the injected fault mechanism.

In [ ]:
print(pd.read_csv(ROOT/'reports'/'forecast_metrics.csv').to_string(index=False))
print(pd.read_csv(ROOT/'reports'/'fault_detection_metrics.csv').to_string(index=False))

## 5. Suggested experiment: feature ablation
Modify `fault_features` in `src/run_analysis.py` to exclude `performance_ratio` and `actual_power_kw`. Re-run the pipeline and compare recall/F1. Explain why this matters for early warning before a power shortfall is fully observed.